# 06 · Category strategy: Kraljic matrix and supplier risk

The Kraljic matrix is computed from the data, item by item:

- **Profit impact:** ABC class A (the items that make up 80% of spend).
- **Supply risk:** mean of four factors scaled 0 to 1, high from 0.5: supplier concentration within the item (1 for a single source), spend-weighted lead time over 90 days, share of spend imported from outside Mercosur, and origin concentration (HHI) of the item's NCM in Comex Stat.

The supplier risk score weighs dependency on items it supplies alone (35%), lead time (25%), quality in PPM (20%) and import exposure (20%). Code in `src/strategy.py`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
from src import gap, strategy
from src.db import DEFAULT_DB, connect, create_schema
from src.figures import kraljic_matrix

con = connect(DEFAULT_DB)
create_schema(con)
materials = strategy.material_risk(con)
print("Origin concentration from Comex Stat year", materials.attrs["market_year"])

## The matrix

In [ ]:
kraljic_matrix(con);

In [ ]:
quadrants = (
    materials.groupby("quadrant")
    .agg(
        items=("material_id", "count"),
        spend_share=("spend_share", "sum"),
        mean_risk=("supply_risk", "mean"),
    )
    .round(3)
)
quadrants["action"] = quadrants.index.map(strategy.ACTIONS)
quadrants

## By family

In [ ]:
strategy.family_summary(materials).round(3)

## What drives supply risk

Mean of each factor by quadrant: which factors put items above the line.

In [ ]:
materials.groupby("quadrant")[list(strategy.FACTORS)].mean().round(3)

## Supplier risk and actions

In [ ]:
risk = strategy.supplier_risk(con)
actions = strategy.supplier_actions(risk, gap.supplier_gaps(con))
print(risk["tier"].value_counts().to_dict())
actions[actions["actions"] != ""][
    [
        "supplier_id",
        "country",
        "tier",
        "risk_score",
        "dependency",
        "lead_time_days",
        "ppm",
        "actions",
    ]
].round(2)

## Reading

- **40% of spend is strategic** (46 items) and another **40% is leverage** (52 items), all harnesses. Strategic items deserve long-term agreements with the quarterly copper clause and cost reviews against should-cost, not yearly price fights; leverage items have a competitive supply base, so annual RFQs against should-cost are where the savings potential is easiest to take.
- **Connectors** sit mostly in **bottleneck** and **non-critical**: small in spend, but single sources and long Asian lead times make them a stoppage risk. Safety stock and a second, closer source matter more here than price.
- **Import exposure** is what separates high-risk items: 95–97% of their spend comes from outside Mercosur, against 30–39% for the low-risk quadrants. Moving volume to Brazilian or Paraguayan plants lowers risk faster than adding suppliers in the same country.
- **14 of 60 suppliers** score as high risk; 3 of them (S035 and S058 in China, S011 in Mexico) hold mostly items they supply alone and should get a second source.
- **S023 and S006**, both Japanese plants, are flagged on price and high risk: the first suppliers to address. **S003** (Morocco) and **S027** (China) are flagged on price with medium risk: a competitive RFQ against should-cost is enough.